In [ ]:
-- Window value functions
-- 1. LEAD() Function : It returns the value from the next row within the Window 
-- LEAD(Sales, 2, 10) OVER(PARTITION BY ProductID ORDER BY OrderDate)
-- Analyze the month-over-month performance by finding the percentage change 
-- in sales between the current and previous months 
SELECT 
*,
currentmonthsales - previousmonthsales as MoM_change,
ROUND(CAST((currentmonthsales - previousmonthsales) as float) / previousmonthsales * 100, 1) AS MoM_PercentageChange
FROM (
    SELECT
        MONTH(OrderDate) AS OrderMonth,
        SUM(Sales) as currentmonthsales,
        LAG(SUM(Sales)) OVER (ORDER BY MONTH(OrderDate)) previousmonthsales
    from sales.orders
    GROUP BY MONTH(OrderDate)
)t 

(3 rows affected)

OrderMonth | currentmonthsales | previousmonthsales | MoM_change | MoM_PercentageChange
-----------+-------------------+--------------------+------------+---------------------
1          | 105               | NULL               | NULL       | NULL                
2          | 195               | 105                | 90         | 85.7                
3          | 80                | 195                | -115       | -59                 
(3 rows)

Total execution time: 00:00:00.035

In [ ]:
-- USE CASE : MAX AND MIN(Customer Retention Analysis) = Measure customer's behaviour and loyalty to help 
-- businesses build strong relationships with customers.
-- Q. Analyze customer loyalty by ranking customers based on the average number of days between orders
SELECT
CustomerID,
AVG(DaysUntilNextOrder) AvgDays
FROM(
    SELECT
        OrderID,
        CustomerID,
        OrderDate AS Currentorder,
        LEAD(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate) Nextorder,
        DATEDIFF(DAY, OrderDate, LEAD(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate)) AS DaysUntilNextOrder
    from Sales.Orders
) t 
GROUP BY CustomerID

(4 rows affected)

CustomerID | AvgDays
-----------+--------
1          | 18     
2          | 34     
3          | 34     
4          | NULL   
(4 rows)

Total execution time: 00:00:00.035

In [ ]:
-- Find the lowest and highest sales for each product 
-- First_value & Last_value 

SELECT
    OrderID,
    ProductID,
    Sales,
    FIRST_VALUE(Sales) OVER (PARTITION BY ProductID ORDER BY sales) LowestSales,
    LAST_VALUE(Sales) OVER (PARTITION BY ProductID ORDER BY sales 
    ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) HighestSales
FROM Sales.Orders

(10 rows affected)

OrderID | ProductID | Sales | LowestSales | HighestSales
--------+-----------+-------+-------------+-------------
8       | 101       | 90    | 10          | 90          
3       | 101       | 20    | 10          | 90          
9       | 101       | 20    | 10          | 90          
1       | 101       | 10    | 10          | 90          
10      | 102       | 60    | 15          | 60          
7       | 102       | 30    | 15          | 60          
2       | 102       | 15    | 15          | 60          
6       | 104       | 50    | 25          | 50          
5       | 104       | 25    | 25          | 50          
4       | 105       | 60    | 60          | 60          
(10 rows)

Total execution time: 00:00:00.034